# Tableau 10 calculé — attributs des bassins d'apport marocains (Colab, GEE compte principal)

Remplace le tableau 10 « indicatif » par un tableau CALCULÉ avec la même chaîne que
les 269 bassins espagnols : HydroSHEDS 03DIR (Afrique) → délinéation pyflwdir →
ERA5-Land 1960–2021 moyenné par polygone → les 8 attributs → verdict d'enveloppe.

⚠ **AVANT Run All : vérifier CHAQUE coordonnée de mur de barrage sur Google Maps**
(cellule 2). Mes valeurs sont des approximations à ±quelques km — un mur mal placé
= un bassin faux. Quand c'est fait, mettre `COORDS_VERIFIEES=True`.

Sorties (Drive `pfe_rainfall/morocco/`) : `catchments_maroc.gpkg`,
`attributes_maroc.csv` (→ tableau 10 final), + verdicts d'enveloppe imprimés.

In [ ]:
!pip -q install earthengine-api pyflwdir rasterio geopandas shapely pyproj >/dev/null
import ee, os, time, glob, numpy as np, pandas as pd, rasterio, geopandas as gpd, pyflwdir
from rasterio.transform import rowcol
from rasterio import features
from shapely.geometry import shape
from shapely.ops import unary_union
ee.Authenticate(); ee.Initialize(project="pfe-rainfall")
from google.colab import drive; drive.mount('/content/drive')
BASE="/content/drive/MyDrive/pfe_rainfall"; MA=os.path.join(BASE,"morocco"); os.makedirs(MA,exist_ok=True)
print("ready")

In [ ]:
# Barrages cibles — lon/lat = MUR du barrage. TOUT VÉRIFIER sur Google Maps avant Run All.
COORDS_VERIFIEES=False   # <<< passer à True après vérification manuelle
DAMS=pd.DataFrame([
 # nom,                    oued,            lon,      lat,     aire publiée ~km² (source à citer)
 ("Al Wahda",              "Ouergha",       -5.291,   34.583,  6200),
 ("Idriss Ier",            "Inaouene",      -4.680,   34.163,  3680),
 ("Bin el Ouidane",        "El Abid",       -6.454,   32.113,  6400),
 ("Ahmed El Hansali",      "Oum Er-Rbia",   -6.033,   32.867,  8420),
 ("Hassan Ier",            "Lakhdar",       -7.070,   31.850,  1650),
 ("Aoulouz",               "Souss",         -8.170,   30.700,  4500),
 ("Mansour Eddahbi",       "Draa",          -6.780,   30.940,  15000),
 ("Mohammed V",            "Moulouya",      -2.960,   34.650,  49920),
 ("Sidi Mohamed Ben Abdellah","Bou Regreg", -6.660,   33.940,  9800),
], columns=["nom","oued","lon","lat","aire_pub_km2"])
assert COORDS_VERIFIEES, "Vérifier les 9 coordonnées sur Google Maps puis mettre COORDS_VERIFIEES=True"
print(DAMS.to_string(index=False))

## 1. HydroSHEDS 03DIR Maroc (une fois, cache) → délinéation

In [ ]:
dir_tif=os.path.join(MA,"hysheds_dir_maroc.tif")
if not os.path.exists(dir_tif):
    img=ee.Image("WWF/HydroSHEDS/03DIR"); region=ee.Geometry.Rectangle([-13.5,27.5,-1.0,36.2])
    t=ee.batch.Export.image.toDrive(image=img.select("b1").toInt16(),description="hysheds_dir_maroc",
        folder="pfe_hysheds_maroc",fileNamePrefix="hysheds_dir_maroc",region=region,scale=92.77,
        crs="EPSG:4326",maxPixels=1e10); t.start()
    while t.status()["state"] not in ("COMPLETED","FAILED","CANCELLED"):
        print(t.status()["state"]); time.sleep(30)
    print("copier hysheds_dir_maroc.tif de MyDrive/pfe_hysheds_maroc/ vers morocco/ puis relancer")
with rasterio.open(dir_tif) as ds:
    dirarr=ds.read(1); transform=ds.transform; nrow,ncol=dirarr.shape
valid={0,1,2,4,8,16,32,64,128}
a=dirarr.astype(np.int32); a[a==255]=0
a=np.where(np.isin(a,list(valid)),a,247).astype(np.uint8)
flw=pyflwdir.from_array(a,ftype="d8",transform=transform,latlon=True,cache=True)
upa=flw.upstream_area(unit="km2"); upa=np.where(upa<0,0,upa).reshape(nrow,ncol)
# SMBA : litige aire publiée (9 770 = vraisemblablement zone de planification Bouregreg-Chaouia)
# -> snap au MAX d'aire amont au droit du mur (comme les barrages espagnols), aire délinéée rapportée,
#    note de bas de tableau. Le verdict d'enveloppe est identique quel que soit le chiffre (> 3 500 km²).
SNAP_MAX_UPA={"Sidi Mohamed Ben Abdellah"}
polys=[]; qc=[]
for r in DAMS.itertuples():
    r0,c0=rowcol(transform,r.lon,r.lat)
    # fenêtre ADAPTATIVE : ±12 cellules (~1,1 km) puis élargie jusqu'à ±45 (~4,2 km)
    # — indispensable pour les grands réservoirs où le mur est loin du thalweg codé.
    mode_max = r.nom in SNAP_MAX_UPA
    best=None; win=12
    while True:
        best=None
        for dr in range(-win,win+1):
            for dc in range(-win,win+1):
                rr,cc=r0+dr,c0+dc
                if 0<=rr<nrow and 0<=cc<ncol and upa[rr,cc]>0:
                    if mode_max:
                        score=-upa[rr,cc]          # max d'aire amont
                    else:
                        score=abs(upa[rr,cc]-r.aire_pub_km2)/r.aire_pub_km2
                    if best is None or score<best[0]: best=(score,rr,cc,upa[rr,cc])
        if mode_max and best is not None and win>=23: break        # ±2,1 km suffisent pour un mur
        if (not mode_max) and ((best is not None and best[0]<0.15) or win>=45): break
        win+=11
    assert best is not None, f"{r.nom}: aucune cellule de rivière à moins de 4 km — coordonnée à revoir"
    _,rr,cc,aa=best
    err=abs(aa-r.aire_pub_km2)/r.aire_pub_km2   # rapporté pour information, même en mode max
    # debug : top-3 candidats dans la fenêtre finale (aide à diagnostiquer un mauvais bras)
    cands=[]
    for dr in range(-win,win+1):
        for dc in range(-win,win+1):
            rr2,cc2=r0+dr,c0+dc
            if 0<=rr2<nrow and 0<=cc2<ncol and upa[rr2,cc2]>50: cands.append(upa[rr2,cc2])
    top=sorted(set(int(x) for x in cands),reverse=True)[:3]
    print(f"{r.nom}: win=±{win} | top upa candidats {top} | retenu {aa:.0f}")
    bas=flw.basins(idxs=np.array([rr*ncol+cc],dtype=np.int64)); m=bas>0
    sh=[shape(s) for s,v in features.shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
    polys.append(dict(nom=r.nom,geometry=unary_union(sh)))
    qc.append(dict(nom=r.nom,aire_pub=r.aire_pub_km2,upa_snap=round(aa,0),err_rel=round(err,3)))
cat=gpd.GeoDataFrame(polys,geometry="geometry",crs="EPSG:4326")
cat["area_km2"]=cat.to_crs("EPSG:3035").area.values/1e6
cat.to_file(os.path.join(MA,"catchments_maroc.gpkg"),driver="GPKG")
print(pd.DataFrame(qc).to_string(index=False))
print("\nerr_rel > 0,25 = coordonnée ou aire publiée suspecte — corriger avant de continuer")
print("(exception SMBA : snap max-upa assumé ; err_rel vs aire publiée rapporté pour la note de bas de tableau)")

## 2. Forçages ERA5-Land 1960–2021 (batch annuel, ~62 petites tâches) + attributs

In [ ]:
RAW_FOLDER="pfe_maroc_forc"; RAW_DIR="/content/drive/MyDrive/"+RAW_FOLDER
cat=gpd.read_file(os.path.join(MA,"catchments_maroc.gpkg"))
cat_s=cat.copy(); cat_s["geometry"]=cat_s.geometry.simplify(0.004)
fc=ee.FeatureCollection([ee.Feature(ee.Geometry(r.geometry.__geo_interface__),{"nom":r["nom"]}) for _,r in cat_s.iterrows()])
era5=ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select(["total_precipitation_sum","temperature_2m_min","temperature_2m_max"])
def export_year(Y):
    coll=era5.filterDate(f"{Y}-01-01",f"{Y+1}-01-01")
    def per_img(img):
        d=img.date().format("YYYY-MM-dd")
        return img.reduceRegions(collection=fc,reducer=ee.Reducer.mean(),scale=11132,tileScale=4).map(lambda f: f.set("date",d))
    t=ee.batch.Export.table.toDrive(collection=coll.map(per_img).flatten(),description=f"maforc_{Y}",
        folder=RAW_FOLDER,fileNamePrefix=f"maforc_{Y}",fileFormat="CSV",
        selectors=["nom","date","total_precipitation_sum","temperature_2m_min","temperature_2m_max"])
    t.start(); return t
tasks={Y:export_year(Y) for Y in range(1960,2022) if not os.path.exists(os.path.join(RAW_DIR,f"maforc_{Y}.csv"))}
print("started",len(tasks))
while tasks:
    done=[y for y,t in tasks.items() if t.status()["state"] in ("COMPLETED","FAILED","CANCELLED")]
    for y in done: tasks.pop(y)
    if tasks: time.sleep(30)
print("exports done")

In [ ]:
drive.mount('/content/drive',force_remount=True)
df=pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(RAW_DIR,"maforc_*.csv")))],ignore_index=True)
df=df.rename(columns={"total_precipitation_sum":"pr_m","temperature_2m_min":"tmin_k","temperature_2m_max":"tmax_k"})
df["date"]=pd.to_datetime(df["date"])
cent=cat.to_crs(3035).geometry.centroid.to_crs(4326); lat_map=dict(zip(cat["nom"],cent.y))
def hargreaves(dates,tmin,tmax,tmean,lat):
    J=pd.to_datetime(dates).dt.dayofyear.values; phi=np.radians(lat)
    dr=1+0.033*np.cos(2*np.pi*J/365.0); dec=0.409*np.sin(2*np.pi*J/365.0-1.39)
    ws=np.arccos(np.clip(-np.tan(phi)*np.tan(dec),-1,1))
    Ra=(24*60/np.pi)*0.0820*dr*(ws*np.sin(phi)*np.sin(dec)+np.cos(phi)*np.cos(dec)*np.sin(ws))
    return np.clip(0.0023*(Ra*0.408)*(tmean+17.8)*np.sqrt(np.clip(tmax-tmin,0,None)),0,None)
rows=[]
for nom,g in df.groupby("nom"):
    g=g.sort_values("date")
    p=pd.to_numeric(g["pr_m"],errors="coerce").values*1000.0
    tmin=pd.to_numeric(g["tmin_k"],errors="coerce").values-273.15
    tmax=pd.to_numeric(g["tmax_k"],errors="coerce").values-273.15
    tm=(tmin+tmax)/2
    pet=hargreaves(g["date"],tmin,tmax,tm,lat_map[nom])
    pm=np.nanmean(p); mon=pd.Series(p,index=g["date"].values).groupby(pd.DatetimeIndex(g["date"]).month).mean()
    rows.append(dict(nom=nom,p_mean=round(pm,3),pet_mean=round(np.nanmean(pet),3),
        aridity=round(np.nanmean(pet)/pm,3),frac_snow=round(float(np.nansum(p[tm<0])/np.nansum(p)),4),
        p_seasonality=round(float((mon.max()-mon.min())/mon.mean()),3)))
clim=pd.DataFrame(rows)
elev=ee.Image("MERIT/Hydro/v1_0_1").select("elv").rename("elev_mean")
slope=ee.Terrain.slope(elev).rename("slope_mean")
forest=ee.ImageCollection("ESA/WorldCover/v200").first().eq(10).rename("forest_frac")
res=elev.addBands([slope,forest]).reduceRegions(collection=fc,reducer=ee.Reducer.mean(),scale=250).map(lambda f:f.setGeometry(None))
gee=pd.DataFrame([d["properties"] for d in res.getInfo()["features"]])
att=clim.merge(gee,on="nom").merge(cat[["nom","area_km2"]],on="nom").round(3)
# verdict d'enveloppe (bornes du chapitre 5 : surface 20-3500, aridité, pérennité non évaluable ici)
def verdict(r):
    v=[]
    if not (20<=r["area_km2"]<=3500): v.append("surface hors domaine")
    if r["aridity"]>2.2: v.append("strate aride (KGE médian négatif)")
    elif r["aridity"]>1.6: v.append("strate intermédiaire (KGE ~0,27)")
    else: v.append("strate humide (KGE ~0,47)")
    return " ; ".join(v)
att["verdict_enveloppe"]=att.apply(verdict,axis=1)
att.to_csv(os.path.join(MA,"attributes_maroc.csv"),index=False)
print(att[["nom","area_km2","aridity","p_mean","verdict_enveloppe"]].to_string(index=False))
print("\n-> attributes_maroc.csv = le tableau 10 CALCULÉ (copier dans le dossier local pour la thèse)")